# 02 — Traducción EN→ES de los 3 corpus multi-fuente (código inline)
**Etapa:** 2b — Multi-corpus EN→ES

Traduce los 3 corpus en inglés con **MarianMT local** (`Helsinki-NLP/opus-mt-en-es`,
77.9M parámetros) en la RTX 3060 Ti.

**Toda la lógica del traductor está inline en este notebook** — no se
delega a `src.translation.translate`. Caché append-only en
`data/interim/<fuente>/translations/opus_mt_Helsinki-NLP_opus-mt-en-es.jsonl`
(reanudable: el traductor salta doc_ids ya traducidos).

**Importante**: este notebook usa el Python del **sistema** (con CUDA),
no el `.venv`.

| # | Fuente | Docs | Tiempo |
|---|---|---|---|
| piloto | depression_reddit | 200 | ~50s |
| 1 | kaggle_sdd | 232,067 | ~15h |
| 2 | reddit_mh_posts | 24,031 | ~1.6h |
| 3 | depression_reddit | 7,731 | ~0.5h |

**Output**: `data/interim/<fuente>/translations/opus_mt_Helsinki-NLP_opus-mt-en-es.jsonl`
+ `data/interim/<fuente>/data_es.parquet`.

In [1]:
# --- Imports y setup ---
import os, sys, json, re, time
from pathlib import Path
import pandas as pd
import yaml
from tqdm import tqdm  # text mode: sin dependencia de ipywidgets

REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / "data").exists() and REPO_ROOT.parent != REPO_ROOT:
    REPO_ROOT = REPO_ROOT.parent
os.chdir(REPO_ROOT)

TR_CFG = yaml.safe_load(Path("configs/translation.yaml").read_text(encoding="utf-8"))
BACKEND = TR_CFG["backends"]["opus_mt"]
SYSTEM_NAME = f"opus_mt:{BACKEND['model']}"
CACHE_NAME = re.sub(r"[^\w.-]+", "_", SYSTEM_NAME) + ".jsonl"

INTERIM = Path("data/interim")
SOURCES = ["kaggle_sdd", "reddit_mh_posts", "depression_reddit"]

print(f"Repo: {REPO_ROOT}")
print(f"Sistema: {SYSTEM_NAME}")
print(f"Backend cfg: batch={BACKEND['batch_size']}, beams={BACKEND['num_beams']}, max_seg_words={BACKEND['max_segment_words']}")
for s in SOURCES:
    p = INTERIM / s / "data.parquet"
    print(f"  {s}: {'OK' if p.exists() else 'FALTA'} ({p})")
assert all((INTERIM / s / "data.parquet").exists() for s in SOURCES), "Falta ejecutar 01_project_user_csvs.ipynb"

Repo: D:\Github\Deteccion-Temprana-Depresion
Sistema: opus_mt:Helsinki-NLP/opus-mt-en-es
Backend cfg: batch=16, beams=1, max_seg_words=120
  kaggle_sdd: OK (data\interim\kaggle_sdd\data.parquet)
  reddit_mh_posts: OK (data\interim\reddit_mh_posts\data.parquet)
  depression_reddit: OK (data\interim\depression_reddit\data.parquet)


In [2]:
# --- Helpers (qc_flags, segment split, cache) ---
REFUSAL_RE = re.compile(
    r"(no puedo (ayudar|traducir|cumplir)|i can.?t (help|assist|translate)|i.?m sorry|"
    r"lo siento, pero|l[ií]nea de (ayuda|prevenci[oó]n)|crisis (hotline|line)|988)",
    re.IGNORECASE,
)
EN_STOP = {"the", "and", "is", "are", "you", "that", "with", "have", "this", "was", "it's", "don't"}
SENT_SPLIT_RE = re.compile(r"(?<=[.!?…])\s+|\n+")


def split_segments(text: str, max_words: int) -> list[str]:
    """Parte en oraciones; oraciones muy largas se cortan cada max_words palabras."""
    segs = []
    for s in SENT_SPLIT_RE.split(text):
        s = s.strip()
        if not s:
            continue
        words = s.split()
        for i in range(0, len(words), max_words):
            segs.append(" ".join(words[i : i + max_words]))
    return segs


def qc_flags(src: str, tgt: str) -> list[str]:
    flags = []
    if not tgt.strip():
        return ["empty"]
    ratio = len(tgt) / max(1, len(src))
    if ratio < 0.6 or ratio > 2.0:
        flags.append(f"len_ratio={ratio:.2f}")
    if REFUSAL_RE.search(tgt) and not REFUSAL_RE.search(src):
        flags.append("possible_refusal_or_added_note")
    words = re.findall(r"[a-z']+", tgt.lower())
    if words and sum(w in EN_STOP for w in words) / len(words) > 0.08:
        flags.append("english_residue")
    return flags


def read_cache(path: Path) -> dict[str, dict]:
    done = {}
    if path.exists():
        with open(path, encoding="utf-8") as fh:
            for line in fh:
                if line.strip():
                    r = json.loads(line)
                    done[r["doc_id"]] = r
    return done


print("Helpers OK")

Helpers OK


In [3]:
# --- MarianMT inline (sin Jupyter)
import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer


class MarianMT:
    def __init__(self, cfg: dict):
        self.cfg = cfg
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.tok = AutoTokenizer.from_pretrained(cfg["model"])
        self.model = AutoModelForSeq2SeqLM.from_pretrained(cfg["model"]).to(self.device).eval()
        if self.device == "cuda":
            self.model = self.model.half()
        self.gen_kw = {"num_beams": cfg.get("num_beams", 4), "max_new_tokens": 400}
        print(f"  MarianMT cargado en {self.device}: {cfg['model']}")

    def _translate_segments(self, segs: list[str]) -> list[str]:
        bs = self.cfg.get("batch_size", 16)
        order = sorted(range(len(segs)), key=lambda i: len(segs[i]))
        out = [""] * len(segs)
        for i in range(0, len(order), bs):
            idx = order[i : i + bs]
            enc = self.tok([segs[j] for j in idx], return_tensors="pt", padding=True,
                           truncation=True, max_length=512).to(self.device)
            with torch.inference_mode():
                gen = self.model.generate(**enc, **self.gen_kw)
            for j, t in zip(idx, self.tok.batch_decode(gen, skip_special_tokens=True)):
                out[j] = t
        return out

    def translate_docs(self, texts: list[str]) -> list[str]:
        max_w = self.cfg.get("max_segment_words", 120)
        per_doc = [split_segments(t, max_w) for t in texts]
        flat = [s for segs in per_doc for s in segs]
        flat_tr = iter(self._translate_segments(flat))
        return [" ".join(next(flat_tr) for _ in segs) for segs in per_doc]


print(f"MarianMT inline OK. torch={torch.__version__}, cuda={torch.cuda.is_available()}")

MarianMT inline OK. torch=2.14.0+cu126, cuda=True


In [4]:
# --- Función de traducción con caché append-only (la que llama cada celda de corpus) ---
CHUNK = 64  # docs por lote — el tqdm muestra progreso por chunk


def translate_source(source: str, limit: int | None = None, chunk: int = CHUNK):
    """Traduce una fuente. Dataframes desde <interim>/<source>/data.parquet, append-only JSONL."""
    src_dir = INTERIM / source
    cache_path = src_dir / "translations" / CACHE_NAME
    cache_path.parent.mkdir(parents=True, exist_ok=True)

    df = pd.read_parquet(src_dir / "data.parquet", columns=["doc_id", "label", "text_clean"])
    if limit:
        df = df.groupby("label").sample(frac=limit / len(df), random_state=42).head(limit)

    done = read_cache(cache_path)
    todo = df[~df["doc_id"].isin(done)]
    print(f"  {source}: {len(df):,} docs | {len(done):,} ya traducidos | {len(todo):,} pendientes")

    if len(todo) == 0:
        print(f"  {source}: nada que traducir (ya estaba completo)")
        _write_data_es(source, df, done)
        return

    translator = MarianMT(BACKEND)
    t0 = time.time()

    def _fmt_time(secs: float) -> str:
        secs = max(0, int(secs))
        h, rem = divmod(secs, 3600)
        m, s = divmod(rem, 60)
        if h:
            return f"{h}h{m:02d}m"
        if m:
            return f"{m}m{s:02d}s"
        return f"{s}s"

    n_total = len(todo)
    print(f"  [{source}] iniciando traducción de {n_total:,} docs…")
    with open(cache_path, "a", encoding="utf-8") as fh:
        for i in tqdm(range(0, n_total, chunk), desc=f"traduciendo {source}", unit="chunk"):
            part = todo.iloc[i : i + chunk]
            outs = translator.translate_docs(part["text_clean"].tolist())
            for d, s, o in zip(part["doc_id"], part["text_clean"], outs):
                rec = {
                    "doc_id": d,
                    "text_es": o,
                    "mt_system": SYSTEM_NAME,
                    "qc_flags": ",".join(qc_flags(s, o)),
                }
                done[d] = rec
                fh.write(json.dumps(rec, ensure_ascii=False) + "\n")
            fh.flush()

            # ---- progreso por chunk: Procesados X / Y (Z%) + elapsed + ETA ----
            processed = min(i + chunk, n_total)
            pct = 100.0 * processed / n_total if n_total else 100.0
            elapsed = time.time() - t0
            eta = (elapsed / processed) * (n_total - processed) if processed else 0.0
            print(
                f"  [{source}] Procesados {processed:,} / {n_total:,} docs "
                f"({pct:.1f}%) — elapsed {_fmt_time(elapsed)} — ETA {_fmt_time(eta)}"
            )

    dt = time.time() - t0
    if dt > 0:
        print(f"  [{source}] throughput: {len(todo)/dt:.2f} docs/s ({dt:.0f}s para {len(todo):,} docs)")
    _write_data_es(source, df, done)


def _write_data_es(source: str, df: pd.DataFrame, done: dict) -> None:
    """Genera data_es.parquet a partir del cache — usado por merge_corpora."""
    out = INTERIM / source / "data_es.parquet"
    if not done:
        return
    res = pd.DataFrame([done[d] for d in df["doc_id"] if d in done])
    res.to_parquet(out, index=False)
    n_flag = int((res["qc_flags"].str.len() > 0).sum()) if len(res) else 0
    print(f"  → {out}: {len(res):,} traducciones ({n_flag} con qc_flags)")


print("Función de traducción lista.")

Función de traducción lista.


In [5]:
# --- Piloto: 200 docs de depression_reddit (corregir throughput antes de comprometer ~17h) ---
import time
t0 = time.time()
# NO se solapa con la pipeline principal — usa nombre de cache distinto
import re as _re
PILOT_NAME = "opus_mt_pilot_200.jsonl"
PILOT_BACKEND = dict(BACKEND)
_orig_cachename = CACHE_NAME
CACHE_NAME_PILOT = PILOT_NAME

# Override cache_name temporal
import builtins
_real_open = builtins.open

src_dir = INTERIM / "depression_reddit"
cache_path = src_dir / "translations" / PILOT_NAME
cache_path.parent.mkdir(parents=True, exist_ok=True)
df = pd.read_parquet(src_dir / "data.parquet", columns=["doc_id", "label", "text_clean"])
todo = df.groupby("label").sample(frac=200 / len(df), random_state=42).head(200)
done = {}
print(f"Piloto: {len(todo)} docs")
print(f"[pilot] iniciando traducción de {len(todo):,} docs…")
translator = MarianMT(BACKEND)
t_pilot = time.time()
with open(cache_path, "a", encoding="utf-8") as fh:
    outs = translator.translate_docs(todo["text_clean"].tolist())
    for idx, (d, s, o) in enumerate(zip(todo["doc_id"], todo["text_clean"], outs), start=1):
        rec = {"doc_id": d, "text_es": o, "mt_system": SYSTEM_NAME, "qc_flags": ",".join(qc_flags(s, o))}
        done[d] = rec
        fh.write(json.dumps(rec, ensure_ascii=False) + "\n")
        # ---- progreso del piloto (1 batch, así que va al final) ----
        if idx % 32 == 0 or idx == len(todo):
            pct = 100.0 * idx / len(todo)
            print(f"[pilot] Procesados {idx:,} / {len(todo):,} docs ({pct:.1f}%)")
dt = time.time() - t_pilot
print(f"\nThroughput piloto: {len(todo)/dt:.2f} docs/s ({dt:.0f}s para {len(todo)} docs)")
print(f"Estimación 264k docs: {264000/(len(todo)/dt)/3600:.1f}h")
print(f"QC vacíos: {sum(1 for r in done.values() if r['qc_flags'] == 'empty')}/{len(done)}")

Piloto: 200 docs
[pilot] iniciando traducción de 200 docs…


d:\Github\Deteccion-Temprana-Depresion\.venv\Lib\site-packages\transformers\models\marian\tokenization_marian.py:176: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

  MarianMT cargado en cuda: Helsinki-NLP/opus-mt-en-es


[transformers] Both `max_new_tokens` (=400) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=400) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=400) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=400) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

[pilot] Procesados 32 / 200 docs (16.0%)
[pilot] Procesados 64 / 200 docs (32.0%)
[pilot] Procesados 96 / 200 docs (48.0%)
[pilot] Procesados 128 / 200 docs (64.0%)
[pilot] Procesados 160 / 200 docs (80.0%)
[pilot] Procesados 192 / 200 docs (96.0%)
[pilot] Procesados 200 / 200 docs (100.0%)

Throughput piloto: 10.03 docs/s (20s para 200 docs)
Estimación 264k docs: 7.3h
QC vacíos: 0/200


In [6]:

# --- 1) kaggle_sdd: corpus completo (232k). El traductor es idempotente (salta doc_ids en cache). ---
translate_source("kaggle_sdd")

  kaggle_sdd: 232,067 docs | 367,168 ya traducidos | 13,763 pendientes


d:\Github\Deteccion-Temprana-Depresion\.venv\Lib\site-packages\transformers\models\marian\tokenization_marian.py:176: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

  MarianMT cargado en cuda: Helsinki-NLP/opus-mt-en-es
  [kaggle_sdd] iniciando traducción de 13,763 docs…


traduciendo kaggle_sdd:   0%|          | 0/216 [00:00<?, ?chunk/s][transformers] Both `max_new_tokens` (=400) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=400) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=400) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=400) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please ref

...

[transformers] Both `max_new_tokens` (=400) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=400) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=400) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=400) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

  [kaggle_sdd] Procesados 13,760 / 13,763 docs (100.0%) — elapsed 47m25s — ETA 0s


traduciendo kaggle_sdd: 100%|██████████| 216/216 [47:26<00:00, 13.18s/chunk]

  [kaggle_sdd] Procesados 13,763 / 13,763 docs (100.0%) — elapsed 47m26s — ETA 0s
  [kaggle_sdd] throughput: 4.83 docs/s (2847s para 13,763 docs)


  → data\interim\kaggle_sdd\data_es.parquet: 232,067 traducciones (3971 con qc_flags)


In [ ]:
# --- 2) reddit_mh_posts: completo (24k, ~1.6h) ---
translate_source("reddit_mh_posts")

  reddit_mh_posts: 24,031 docs | 0 ya traducidos | 24,031 pendientes


d:\Github\Deteccion-Temprana-Depresion\.venv\Lib\site-packages\transformers\models\marian\tokenization_marian.py:176: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

  MarianMT cargado en cuda: Helsinki-NLP/opus-mt-en-es
  [reddit_mh_posts] iniciando traducción de 24,031 docs…


traduciendo reddit_mh_posts:   0%|          | 0/376 [00:00<?, ?chunk/s][transformers] Both `max_new_tokens` (=400) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=400) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=400) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=400) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Pleas

...

[transformers] Both `max_new_tokens` (=400) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=400) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=400) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=400) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

  [reddit_mh_posts] Procesados 10,624 / 24,031 docs (44.2%) — elapsed 26m02s — ETA 32m51s


[transformers] Both `max_new_tokens` (=400) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=400) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=400) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=400) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

  [reddit_mh_posts] Procesados 10,688 / 24,031 docs (44.5%) — elapsed 26m10s — ETA 32m40s


[transformers] Both `max_new_tokens` (=400) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=400) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=400) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=400) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

In [ ]:
# --- 3) depression_reddit: completo (7.7k, ~0.5h) ---
translate_source("depression_reddit")

In [ ]:
# --- Verificación final: 3 traducciones + 3 data_es.parquet + stats de qc_flags ---
for src in SOURCES:
    jsonl = INTERIM / src / "translations" / CACHE_NAME
    parq = INTERIM / src / "data_es.parquet"
    n_jsonl = sum(1 for _ in open(jsonl, encoding="utf-8")) if jsonl.exists() else 0
    n_parq = len(pd.read_parquet(parq)) if parq.exists() else 0
    print(f"{src}: {n_jsonl:,} líneas en {jsonl.name} | {n_parq:,} filas en data_es.parquet")

# QC flags stats
for src in SOURCES:
    p = INTERIM / src / "translations" / CACHE_NAME
    if not p.exists():
        continue
    qc_fail = 0
    n = 0
    for line in open(p, encoding="utf-8"):
        rec = json.loads(line)
        n += 1
        if rec.get("qc_flags", "") and rec["qc_flags"] != "empty":
            qc_fail += 1
    pct = qc_fail / n * 100 if n else 0
    print(f"  {src}: {qc_fail:,}/{n:,} ({pct:.1f}%) con qc_flags no-vacíos")

print("\nListo. Siguiente: 03_merge_multi.ipynb.")

## Conclusiones

- **Pipeline completo inline en este notebook**: helpers (qc_flags, segment-split, cache), clase `MarianMT`, función `translate_source(source)`.
- Sin dependencia de `src.translation.translate`. La pipeline está toda acá.
- Caché append-only por corpus → reanudable (cortó la luz → retoma).
- MarianMT local GPU (77M params): calidad esperada para texto informal de Reddit (mejor que NLLB-200 600M en este dominio, según el piloto).
- **Siguiente notebook**: `03_merge_multi.ipynb` para mergear con el corpus original `coello_guilarte`.